# Extending the Keplerian model

In this tutorial, we show how to extend the base Keplerian model in `harv` to account for potential zero-point offsets between RV surveys or instruments, to complexify the noise model by adding a jitter or Gaussian process background model, or to simultaneously model long-term trends in the data. 

Each of these is supported via a {py:class}`harv.AbstractExtension` that plugs into the same `RejectionSampler` workflow we used in the {doc}`getting started tutorial <0-getting-started>`. We recommend reading that tutorial first if you are new to `harv`.

This tutorial assumes familiarity with the basic `RVData` / `RejectionPrior` / `RejectionSampler` workflow introduced in the previous tutorials.

In [ ]:
import astropy.table as at
import jax
import matplotlib.pyplot as plt
import numpyro.distributions as dist
from unxt import Q

import harv

jax.config.update("jax_enable_x64", True)

%matplotlib inline

## 1. Multi-survey data with instrumental offsets

When two spectrographs observe the same star, the reported radial velocities may differ by an offset because of differences in the wavelength solution or other instrumental effects. If we naively concatenate the two datasets and fit a single Keplerian model, this offset will bias the inferred parameters (especially the systemic velocity and eccentricity) and could lead to poor performance of the sampler.

`harv` supports simultaneously inferring instrumental offsets through the {py:class}`~harv.extensions.MultiSurveyOffset` extension, which adds one linear parameter per instrument (beyond the adopted reference instrument). That is, you specify which survey or dataset should be treated as the reference, and infer a per-instrument additive offset that is analytically marginalized along with `v_sys` and `rv_semiamp`. 

For this example we'll use simulated data so we know the true parameters. We'll imagine that we have two surveys, "survey1" and "survey2", that observed the same star. We can load the data for each survey and store them together in a single `SourceData` object:

In [ ]:
tbl = at.Table.read("../data/simulated-multi-survey-data.fits")
survey1_data = harv.RVData(
    time=Q(tbl["time"][tbl["survey"] == "survey1"].astype("f8"), "day"),
    rv=Q(tbl["rv"][tbl["survey"] == "survey1"].astype("f8"), "km/s"),
    rv_err=Q(tbl["rv_err"][tbl["survey"] == "survey1"].astype("f8"), "km/s"),
)

survey2_data = harv.RVData(
    time=Q(tbl["time"][tbl["survey"] == "survey2"].astype("f8"), "day"),
    rv=Q(tbl["rv"][tbl["survey"] == "survey2"].astype("f8"), "km/s"),
    rv_err=Q(tbl["rv_err"][tbl["survey"] == "survey2"].astype("f8"), "km/s"),
)

source_data = harv.SourceData(survey1=survey1_data, survey2=survey2_data)
source_data

{py:meth}`SourceData.plot <harv.SourceData.plot>` draws each component on a single axes with distinct colors and a legend, so we can see at a glance whether the two datasets agree on the systemic velocity and the orbital signal.

In [ ]:
_ = source_data.plot()

### Building the prior and the model

The `MultiSurveyOffset` extension needs a precomputed *indicator matrix* that says which observation came from which instrument. The {py:meth}`SourceData.indicator_data_by_type <harv.SourceData.indicator_data_by_type>` helper does this for us — given a reference instrument, it returns:

- `stacked` — a single `RVData` with all observations concatenated,
- `indicator` — a `(n_obs, n_non_reference)` 0/1 matrix with one column per non-reference instrument, and
- `instrument_names` — the column ordering for `indicator`.

We pick `apogee` as the reference (its offset is absorbed by `v_sys`) and put a Gaussian prior on the HARPS offset. The simplest way to declare the offset prior is to pass it as a keyword argument to {py:meth}`~harv.RejectionPrior.default_rv` whose name matches the non-reference instrument:

In [ ]:
stacked, indicator, instrument_names = source_data.indicator_data_by_type(
    harv.RVData, reference="survey1"
)

prior = harv.RejectionPrior.default_rv(
    period_min=Q(1.0, "day"),
    period_max=Q(3000.0, "day"),
    sigma_K0=Q(30.0, "km/s"),
    sigma_v0=Q(50.0, "km/s"),
    # One offset prior per non-reference instrument. The keyword name must
    # match the instrument name passed to SourceData(...).
    survey2=harv.QD(dist.Normal(0.0, 1.0), "km/s"),
)

Because the offset is a *linear* extension parameter, the rejection sampler needs to know about it inside the model's `linear_prior` so that it gets analytically marginalized along with `rv_semiamp` and `v_sys`. We build the model directly and merge the offset prior into `linear_prior`, then construct the sampler via {py:meth}`~harv.RejectionSampler.from_model`:

In [ ]:
merged_linear = {**prior.linear_prior, **prior.extension_priors}
model = harv.RVModel(
    data=stacked,
    linear_prior=merged_linear,
    extensions=(harv.MultiSurveyOffset(indicator, instrument_names, "km/s"),),
)

sampler = harv.RejectionSampler.from_model(model, prior)
samples = sampler.run(n_prior_samples=10_000_000, max_posterior_samples=1024, seed=42)
samples

The returned `Samples` object now has an extra entry for each non-reference instrument's offset (`harps` in this case), in addition to the standard orbital parameters:

TODO: bug - samples doesn't contain the offset parameter(s) if the prior has the wrong name (e.g., harps instead of "survey2")

In [ ]:
samples.keys()

In [ ]:
axes = samples.wrap_angles().plot_corner(
    ["period", "eccentricity", "rv_semiamp", "v_sys", "survey2"],
)

{py:func}`harv.plot.plot_rv` recognizes the `MultiSurveyOffset` extension when the sampler's extensions are passed in: it will plot each instrument in its own color and shift the non-reference data points by the posterior median offset (`apply_median_offsets=True`, the default).

BUG: apply_median_offsets doesn't seem to do anything True or False?

In [ ]:
ax = harv.plot.plot_rv(
    samples,
    data=source_data,
    extensions=sampler.extensions,
    n_samples=128,
    relative_to_t_ref=True,
    relative_to_median_v_sys=True,
    apply_median_offsets=False,
)

In [ ]:
ax = harv.plot.plot_rv(
    samples,
    data=source_data,
    extensions=sampler.extensions,
    n_samples=128,
    relative_to_t_ref=True,
    relative_to_median_v_sys=True,
    apply_median_offsets=True,
)

## 2. Jitter: handling underestimated uncertainties

Reported per-observation RV uncertainties are often optimistic — pipelines may not propagate every systematic, and astrophysical noise (granulation, p-mode oscillations, low-amplitude spots) is not captured by the formal photon-noise error bar. If we trust the formal errors, the model is forced to fit every fluctuation as a real Keplerian signal, and we get spurious tight constraints on the orbital parameters and bad reduced-χ².

The standard remedy is to add a constant *jitter* term in quadrature to the per-observation variance:

$$\sigma_\mathrm{eff}^2 = \sigma_\mathrm{obs}^2 + s^2$$

where $s$ is sampled jointly with the other parameters. In `harv` this is the {py:class}`~harv.Jitter` extension. We declare a prior on `jitter` (typically a `HalfNormal` since it is a positive scale) and pass the extension to the sampler.

We'll demonstrate on a target whose APOGEE error bars look optimistic relative to the scatter of the data — this is common for bright targets where APOGEE quotes errors of 50–100 m/s but the true epoch-to-epoch scatter is 200–500 m/s.

In [ ]:
# TODO: pick an APOGEE target for which the per-visit VRELERR values are
# noticeably smaller than the visit-to-visit scatter (e.g. a moderately
# bright RGB or main-sequence star without a clear large-amplitude orbit).
# Save the visit RVs to docs/tutorials/data/apogeedr17-<2MASSID>.csv.
tbl = at.Table.read("../data/apogeedr17-<TODO-jitter>.csv")
data = harv.RVData(
    time=Q(tbl["JD"].astype(fdtype), "day"),
    rv=Q(tbl["VHELIO"].astype(fdtype), "km/s"),
    rv_err=Q(tbl["VRELERR"].astype(fdtype), "km/s"),
)
_ = data.plot(relative_to_t_ref=True)

To enable jitter we pass two pieces:

1. a prior named `jitter` to `default_rv` — `HalfNormal` with a scale comparable to the typical RV uncertainty is a sensible default, and
2. a `Jitter(param_unit="km/s")` extension to the sampler.

In [ ]:
prior_jitter = harv.RejectionPrior.default_rv(
    period_min=Q(0.5, "day"),
    period_max=Q(3000.0, "day"),
    sigma_K0=Q(30.0, "km/s"),
    sigma_v0=Q(50.0, "km/s"),
    jitter=harv.QD(dist.HalfNormal(0.5), "km/s"),
)
sampler_jitter = harv.RejectionSampler(
    prior_jitter,
    extensions=(harv.Jitter(param_unit="km/s"),),
)
samples_jitter = sampler_jitter.run(
    data, n_prior_samples=10_000_000, max_posterior_samples=1024, seed=42
)
len(samples_jitter)

Let's compare to a fit that does *not* include jitter, to see how the inferred orbital parameters change:

In [ ]:
prior_no_jitter = harv.RejectionPrior.default_rv(
    period_min=Q(0.5, "day"),
    period_max=Q(3000.0, "day"),
    sigma_K0=Q(30.0, "km/s"),
    sigma_v0=Q(50.0, "km/s"),
)
samples_no_jitter = harv.RejectionSampler(prior_no_jitter).run(
    data, n_prior_samples=10_000_000, max_posterior_samples=1024, seed=42
)
len(samples_no_jitter)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

for ax, (name, s) in zip(
    axes,
    {"no jitter": samples_no_jitter, "with jitter": samples_jitter}.items(),
    strict=True,
):
    ax.plot(s["period"].to_value("day"), s["eccentricity"], ".", alpha=0.4)
    ax.set(xscale="log", xlabel="period [day]", title=name)
axes[0].set_ylabel("eccentricity")

The marginal posterior on the jitter itself is informative: if it pushes against zero, the formal errors are fine; if it concentrates well above zero, that indicates the formal errors are missing real noise.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(samples_jitter["jitter"].to_value("km/s"), bins=32)
ax.set(xlabel="jitter [km/s]", ylabel="posterior count")

## 3. Gaussian process for stellar variability

Jitter is the simplest noise extension, but it assumes the excess noise is *uncorrelated* in time. Many real noise sources are not — red giants, in particular, show RV variability on timescales of hours to weeks driven by convective granulation, surface oscillations, and (for cooler giants) magnetic activity. This produces correlated residuals: nearby epochs are more similar than distant ones. Treating that with a single jitter parameter inflates the error budget too much on long timescales and not enough on short ones.

The {py:class}`~harv.GP` extension uses a Gaussian process to model these correlations as part of the likelihood. The GP kernel adds a covariance matrix $K(t, t')$ to the data covariance, and the linear marginalization machinery still works because the covariance enters the Gaussian likelihood the same way. `harv.GP` accepts any kernel-builder callable that returns an object with an `.evaluate(t, t')` method — `tinygp` is the natural choice.

For this section we use an APOGEE red-giant-branch (RGB) target. RGB stars exhibit characteristic granulation noise on timescales of a few days, so an `ExpSquared` kernel with a few-day length scale is a reasonable starting point.

In [ ]:
# TODO: pick an APOGEE RGB star with enough visits to constrain a GP
# (>= ~12 visits, ideally with some pairs of visits within a day or two so
# the short timescale is sampled). Save the visit RVs to
# docs/tutorials/data/apogeedr17-<2MASSID>.csv.
tbl = at.Table.read("../data/apogeedr17-<TODO-rgb>.csv")
rgb_data = harv.RVData(
    time=Q(tbl["JD"].astype(fdtype), "day"),
    rv=Q(tbl["VHELIO"].astype(fdtype), "km/s"),
    rv_err=Q(tbl["VRELERR"].astype(fdtype), "km/s"),
)
_ = rgb_data.plot(relative_to_t_ref=True)

### Building the GP extension

The `GP` extension needs:

- a `kernel_builder(hp)` callable that takes a dict of unit-stripped hyperparameter values and returns a `tinygp` kernel;
- a list of `ParamInfo` declarations for the hyperparameters (these become *nonlinear* parameters that the rejection sampler draws from their priors); and
- a `time_unit` string telling the extension what unit to strip times to before evaluating the kernel.

A typical RGB-granulation parameterization uses an amplitude `gp_amp` (in the RV units, km/s) and a length scale `gp_scale` (in days):

In [ ]:
import tinygp  # optional dependency

In [ ]:
def rgb_kernel(hp):
    return hp["gp_amp"] ** 2 * tinygp.kernels.ExpSquared(scale=hp["gp_scale"])


gp_ext = harv.GP(
    kernel_builder=rgb_kernel,
    hyperparams=(
        harv.ParamInfo("gp_amp", "km/s"),
        harv.ParamInfo("gp_scale", "day"),
    ),
    time_unit="day",
)

The hyperparameter priors get attached to the prior the same way as `jitter` — by passing them as kwargs to `default_rv`. Use `HalfNormal` for the amplitude (it must be positive) and a `LogUniform` for the length scale (we are typically uncertain about timescale by an order of magnitude or more).

In [ ]:
prior_gp = harv.RejectionPrior.default_rv(
    period_min=Q(1.0, "day"),
    period_max=Q(3000.0, "day"),
    sigma_K0=Q(30.0, "km/s"),
    sigma_v0=Q(50.0, "km/s"),
    gp_amp=harv.QD(dist.HalfNormal(0.5), "km/s"),
    gp_scale=harv.QD(dist.LogUniform(1.0, 100.0), "day"),
)

sampler_gp = harv.RejectionSampler(prior_gp, extensions=(gp_ext,))
samples_gp = sampler_gp.run(
    rgb_data, n_prior_samples=2_000_000, max_posterior_samples=512, seed=42
)
len(samples_gp)

We can include the GP hyperparameters in the corner plot to inspect their posteriors alongside the orbital parameters:

In [ ]:
axes = samples_gp.wrap_angles().plot_corner(
    ["period", "eccentricity", "rv_semiamp", "gp_amp", "gp_scale"],
    marginals=False,
    figsize=(12, 12),
    scatter_kwargs={"alpha": 0.5},
)
axes[1, 0].set_xscale("log")
axes[-1, -2].set_xscale("log")

When we pass the GP extension to `harv.plot.plot_rv`, the time-domain panel shows the GP's conditional mean prediction in addition to the Keplerian orbit overlays — this lets us see how much of the residual structure is being absorbed by the GP versus the orbit:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax = harv.plot.plot_rv(
    samples_gp,
    data=rgb_data,
    extensions=sampler_gp.extensions,
    n_samples=64,
    relative_to_t_ref=True,
    relative_to_median_v_sys=True,
    ax=ax,
)

## 4. Long-term polynomial trend

A common configuration is a *short-period* binary with an outer, long-period companion (or a more distant tertiary). On a baseline shorter than the outer orbit's period, the outer companion produces a roughly linear (or quadratic) drift in the systemic velocity that the inner Keplerian model can't absorb. If we fit only a single Keplerian, the linear trend leaks into a too-eccentric or too-long-period inner orbit.

The {py:class}`~harv.MonomialTrend` extension adds polynomial terms `(t - t_ref)^k` for `k = 1..order` as additional *linear* parameters of the design matrix (so they're analytically marginalized just like `v_sys`). The names of the trend parameters are `trend_1`, `trend_2`, ....

For this section we want a target with a clear short-period RV signal and a long-baseline drift. APOGEE plus HARPS coverage often yields exactly this configuration: HARPS pins down the short-period orbit at a few-night cadence, while APOGEE provides multi-year leverage that reveals the trend.

In [ ]:
# TODO: pick a source with both (a) a clear short-period RV variation and
# (b) a multi-year linear or quadratic drift. Examples to scout for:
#   - APOGEE source flagged with a non-zero VHELIO_AVG slope across visits.
#   - HARPS triple system (e.g. HD 13724-like) with published outer companion.
# Save the visit RVs to docs/tutorials/data/<filename>.csv.
tbl = at.Table.read("../data/<TODO-trend>.csv")
trend_data = harv.RVData(
    time=Q(tbl["JD"].astype(fdtype), "day"),
    rv=Q(tbl["RV"].astype(fdtype), "km/s"),
    rv_err=Q(tbl["RV_ERR"].astype(fdtype), "km/s"),
)
_ = trend_data.plot(relative_to_t_ref=True)

The `MonomialTrend` extension takes the polynomial `order` (1 for a linear drift, 2 to additionally allow curvature) and a `time_unit` for unit-stripping. Because the trend parameters are linear, we declare priors on them via the same `default_rv(...)` kwarg mechanism:

In [ ]:
trend = harv.MonomialTrend(order=1, time_unit="day", obs_unit="km/s")

prior_trend = harv.RejectionPrior.default_rv(
    period_min=Q(0.5, "day"),
    period_max=Q(50.0, "day"),  # short-period inner orbit
    sigma_K0=Q(30.0, "km/s"),
    sigma_v0=Q(50.0, "km/s"),
    trend_1=harv.QD(dist.Normal(0.0, 0.01), "km/s"),  # km/s per day
)

sampler_trend = harv.RejectionSampler(prior_trend, extensions=(trend,))
samples_trend = sampler_trend.run(
    trend_data, n_prior_samples=2_000_000, max_posterior_samples=512, seed=42
)
len(samples_trend)

The corner plot now includes `trend_1` (the linear drift coefficient in km/s/day) alongside the orbital parameters. A nonzero `trend_1` posterior indicates real long-term evolution beyond the short-period orbit:

In [ ]:
axes = samples_trend.wrap_angles().plot_corner(
    ["period", "eccentricity", "rv_semiamp", "v_sys", "trend_1"],
    marginals=False,
    figsize=(12, 12),
    scatter_kwargs={"alpha": 0.5},
)
axes[1, 0].set_xscale("log")

To see what the trend buys us, we can also fit *without* the trend extension and compare the inferred orbital parameters. Without the trend, samples typically migrate to longer periods and/or higher eccentricities to absorb the linear drift:

In [ ]:
prior_no_trend = harv.RejectionPrior.default_rv(
    period_min=Q(0.5, "day"),
    period_max=Q(50.0, "day"),
    sigma_K0=Q(30.0, "km/s"),
    sigma_v0=Q(50.0, "km/s"),
)
samples_no_trend = harv.RejectionSampler(prior_no_trend).run(
    trend_data, n_prior_samples=2_000_000, max_posterior_samples=512, seed=42
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, (name, s) in zip(
    axes,
    {"no trend": samples_no_trend, "with linear trend": samples_trend}.items(),
    strict=True,
):
    ax.plot(s["period"].to_value("day"), s["eccentricity"], ".", alpha=0.4)
    ax.set(xscale="log", xlabel="period [day]", title=name)
axes[0].set_ylabel("eccentricity")

## Combining extensions

All four extensions compose: pass a tuple of extensions to the sampler in the order you want them applied. For example, a multi-survey fit with both a per-instrument offset, a global jitter, and a long-term linear trend is just

```python
extensions = (
    harv.MultiSurveyOffset(indicator, instrument_names, "km/s"),
    harv.Jitter(param_unit="km/s"),
    harv.MonomialTrend(order=1, time_unit="day", obs_unit="km/s"),
)
```

with matching `jitter=...`, `trend_1=...`, and `<instrument>=...` priors passed as kwargs to `default_rv`. The {py:class}`~harv.AbstractExtension` interface is intentionally minimal — only `extra_params`, `modify_design_matrix`, and `modify_covariance` — so writing a new extension (e.g. a custom kernel, an alternative trend basis, or a calibration term tied to an external regressor) is straightforward and slots into exactly the same sampler workflow shown here.